In [1]:
from pyspark.sql import SparkSession
import spark

In [2]:
spark = SparkSession.builder \
        .appName("SparkTable Demo")\
        .enableHiveSupport() \
        .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/13 14:23:09 INFO SparkEnv: Registering MapOutputTracker
26/04/13 14:23:09 INFO SparkEnv: Registering BlockManagerMaster
26/04/13 14:23:09 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/13 14:23:10 INFO SparkEnv: Registering OutputCommitCoordinator


In [3]:
!hadoop fs -ls /tmp/

Found 7 items
-rw-r--r--   2 sowapipython hadoop   10528211 2026-04-01 18:47 /tmp/customers_10mb.csv
-rw-r--r--   2 sowapipython hadoop  168541068 2026-04-01 18:48 /tmp/customers_150mb.csv
-rw-r--r--   2 sowapipython hadoop    1060750 2026-04-01 18:47 /tmp/customers_1mb.csv
-rw-r--r--   2 sowapipython hadoop  343317147 2026-04-01 18:48 /tmp/customers_300mb.csv
drwxrwxrwt   - hdfs         hadoop          0 2026-03-31 14:46 /tmp/hadoop-yarn
drwx-wx-wx   - hive         hadoop          0 2026-03-31 14:47 /tmp/hive
-rw-r--r--   2 root         hadoop         84 2026-04-01 17:13 /tmp/inputhdfsdbz.txt


In [4]:
df = spark.read \
.format ('csv') \
.option('header','True') \
.option('inferSchema','True') \
.load('/tmp/customers_150mb.csv')

In [5]:
df.show(10)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|  Kolkata|  Karnataka|  India|       2023-12-21|    false|
|          1|Customer_1|Ahmedabad|  Telangana|  India|       2023-03-07|     true|
|          2|Customer_2|Bangalore|  Karnataka|  India|       2023-01-14|    false|
|          3|Customer_3|  Chennai|  Karnataka|  India|       2023-11-16|    false|
|          4|Customer_4|    Delhi|  Telangana|  India|       2023-05-23|     true|
|          5|Customer_5|Hyderabad|Maharashtra|  India|       2023-07-10|    false|
|          6|Customer_6|     Pune|West Bengal|  India|       2023-03-04|     true|
|          7|Customer_7|Hyderabad|West Bengal|  India|       2023-08-11|    false|
|          8|Customer_8|  Chennai|West Bengal|  India|       2023-10-21|    false|
|   

In [6]:
spark.sql('show tables').show()

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used


+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



#### Creating a spark table from a DataFrame

#### 1. Temporary Table

In [7]:
# creating a new table as 'temp_customers' using TEMPORARY TABLE
df.createOrReplaceTempView('temp_customers')

In [8]:
spark.sql('show tables').show()

+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|         |temp_customers|       true|
+---------+--------------+-----------+



In [9]:
spark.sql('select * from temp_customers limit 5')

DataFrame[customer_id: int, name: string, city: string, state: string, country: string, registration_date: date, is_active: boolean]

In [10]:
spark.sql('select * from temp_customers limit 5').show()

+-----------+----------+---------+---------+-------+-----------------+---------+
|customer_id|      name|     city|    state|country|registration_date|is_active|
+-----------+----------+---------+---------+-------+-----------------+---------+
|          0|Customer_0|  Kolkata|Karnataka|  India|       2023-12-21|    false|
|          1|Customer_1|Ahmedabad|Telangana|  India|       2023-03-07|     true|
|          2|Customer_2|Bangalore|Karnataka|  India|       2023-01-14|    false|
|          3|Customer_3|  Chennai|Karnataka|  India|       2023-11-16|    false|
|          4|Customer_4|    Delhi|Telangana|  India|       2023-05-23|     true|
+-----------+----------+---------+---------+-------+-----------------+---------+



#### If I create the new spark session and try to see the table temp_customers: I am unable to find it because it exists for that duration only.

In [11]:
spark_new = spark.newSession()

In [12]:
spark_new.sql('show tables').show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [ ]:
## Will get error:
spark_new.sql('select * from temp_customers limit 5').show()

### 2.Global Temporary Table

In [13]:
## creating a new table as 'temp_customers' with GLOBAL TEMPORARY TABLE
df.createOrReplaceGlobalTempView('global_customers')

In [14]:
spark.sql('show tables').show()

+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|         |temp_customers|       true|
+---------+--------------+-----------+



In [15]:
spark.sql('show tables in global_temp').show()

+-----------+----------------+-----------+
|  namespace|       tableName|isTemporary|
+-----------+----------------+-----------+
|global_temp|global_customers|       true|
|           |  temp_customers|       true|
+-----------+----------------+-----------+



In [16]:
spark.sql('select * from global_temp.global_customers limit 5').show()

+-----------+----------+---------+---------+-------+-----------------+---------+
|customer_id|      name|     city|    state|country|registration_date|is_active|
+-----------+----------+---------+---------+-------+-----------------+---------+
|          0|Customer_0|  Kolkata|Karnataka|  India|       2023-12-21|    false|
|          1|Customer_1|Ahmedabad|Telangana|  India|       2023-03-07|     true|
|          2|Customer_2|Bangalore|Karnataka|  India|       2023-01-14|    false|
|          3|Customer_3|  Chennai|Karnataka|  India|       2023-11-16|    false|
|          4|Customer_4|    Delhi|Telangana|  India|       2023-05-23|     true|
+-----------+----------+---------+---------+-------+-----------------+---------+



### 3. Persistent Table

In [ ]:
# Above we didn't save it in disk; those tables are always in in-memory.

In [17]:
df.write.mode('overwrite').saveAsTable('customers_persistent') #Temporary is false; meaning it's permanent table

26/04/13 15:05:27 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


In [18]:
spark.sql('show tables').show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|  default|customers_persistent|      false|
|         |      temp_customers|       true|
+---------+--------------------+-----------+



In [19]:
spark.sql('describe customers_persistent').show()

+-----------------+---------+-------+
|         col_name|data_type|comment|
+-----------------+---------+-------+
|      customer_id|      int|   NULL|
|             name|   string|   NULL|
|             city|   string|   NULL|
|            state|   string|   NULL|
|          country|   string|   NULL|
|registration_date|     date|   NULL|
|        is_active|  boolean|   NULL|
+-----------------+---------+-------+



In [20]:
spark.sql('describe extended customers_persistent').show(truncate = False)

+----------------------------+--------------------------------------------------------------+-------+
|col_name                    |data_type                                                     |comment|
+----------------------------+--------------------------------------------------------------+-------+
|customer_id                 |int                                                           |NULL   |
|name                        |string                                                        |NULL   |
|city                        |string                                                        |NULL   |
|state                       |string                                                        |NULL   |
|country                     |string                                                        |NULL   |
|registration_date           |date                                                          |NULL   |
|is_active                   |boolean                                             

In [21]:
spark.stop()